In [1]:
# 1. Install required packages
# Run this once in your environment
# pip install kaggle pandas sqlite3

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
# %pip install seaborn

In [10]:
#print(os.getcwd())

In [3]:
# 3. Load CSV into pandas

df_ops = pd.read_csv('Walmart_Sales.csv')

In [4]:
df_ops.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [5]:
conn = sqlite3.connect('Walmart_sales.db')
df_ops.to_sql('Walmart_sales', conn, index=False, if_exists='replace')

6435

In [6]:
def query_and_print(sql):
    result = pd.read_sql(sql, conn)
    print(result, '\n')

In [7]:
analysis_queries = {
    'Task 1: Weekly Sales': '''
     SELECT Store, Date, Weekly_Sales,
     SUM(Weekly_Sales) OVER(PARTITION BY Store ORDER BY Date) AS Cumulative_Weekly_Sales
     FROM walmart_sales
     ORDER BY Store, Date;
     ''',

    'Task 2: 7D Rolling Average Sales': '''
     SELECT Store, Date, Weekly_Sales,
     AVG(Weekly_Sales) OVER(
     PARTITION BY Store
     ORDER BY Date
     ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS Rolling_Avg_Weekly_Sales, Holiday_Flag
     FROM walmart_sales
     ORDER BY Store, Date;
    ''',

     'Task 3: Weekly sales holidays vs non-holidays': '''
     SELECT AVG(Weekly_Sales) as avg_weekly_sales, Holiday_Flag,
           CASE  WHEN Holiday_Flag = 1 THEN 'Holiday'
        	     WHEN Holiday_Flag = 0 THEN 'Non-Holiday'
           END AS Holiday_Type
     FROM walmart_sales
     GROUP BY Holiday_Flag;
    ''',
   
    'Task 3: peak sales period per year': '''
    WITH peak_sales AS (
      SELECT 
        MAX(Weekly_Sales) AS Peak_Sales,
        substr(Date, 7, 4) AS Year
      FROM walmart_sales
      GROUP BY Year
    )
    SELECT 
      p.Peak_Sales, 
      p.Year, 
      w.Date AS Peak_Period
    FROM peak_sales p
    JOIN walmart_sales w
      ON p.Peak_Sales = w.Weekly_Sales
      AND p.Year = substr(w.Date, 7, 4)
    ORDER BY w.Date;
    '''
}

In [8]:
for desc, sql in analysis_queries.items():
    print(f'-- {desc} --')
    query_and_print(sql)

-- Task 1: Weekly Sales --
      Store        Date  Weekly_Sales  Cumulative_Weekly_Sales
0         1  01-04-2011    1495064.75             1.495065e+06
1         1  01-06-2012    1624477.58             3.119542e+06
2         1  01-07-2011    1488538.09             4.608080e+06
3         1  01-10-2010    1453329.50             6.061410e+06
4         1  02-03-2012    1688420.76             7.749831e+06
...     ...         ...           ...                      ...
6430     45  30-07-2010     716859.27             1.094135e+08
6431     45  30-09-2011     698986.34             1.101125e+08
6432     45  30-12-2011     869403.63             1.109819e+08
6433     45  31-08-2012     734297.87             1.117162e+08
6434     45  31-12-2010     679156.20             1.123953e+08

[6435 rows x 4 columns] 

-- Task 2: 7D Rolling Average Sales --
      Store        Date  Weekly_Sales  Rolling_Avg_Weekly_Sales  Holiday_Flag
0         1  01-04-2011    1495064.75              1.495065e+06          

In [9]:
conn.close()